# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedkhaled600/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

***Answer:***

**Logistic Regression first, then Random Forest — both evaluated at precision@K, not accuracy.**

My question is "which pages first?" — a ranking question — so per the toolkit, what matters is a
classifier's *probability* evaluated at precision@K, not a bare yes/no label. I start with
Logistic Regression because it's readable (I can name every coefficient and defend it), then add
Random Forest as the stronger candidate, since the ML-05 signal audit showed no single feature
cleanly separates decliners — a tree ensemble can combine several weak, tangled signals the way a
straight line can't. I'm deliberately not reaching for Gradient Boosting this week: the honest
comparison is baseline vs. simple model vs. slightly-stronger model, and adding a third, heavier
model only earns its place if the first two don't already answer the question — which, per the
results below, they do.

Both models reuse the exact same 10-feature vector, label, and `month=2026-03` slice as ML-05/
ML-06, so this is a fair fight against my own baseline, not a new dataset.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb
import duckdb, pandas as pd, numpy as np
con = duckdb.connect()

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "fact_content_daily_performance/month=2026-03/*.parquet"
CUTOFF = "DATE '2026-03-15'"
SEED = 42  # fixed for reproducibility -- rerunning should reproduce the same table below

# --- Same panel construction as ML-05/ML-06: h1 features, h2 label, no future-window inputs ---
panel = con.sql(f"""
    WITH h1 AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks)       AS prior_clicks_h1,
               SUM(gsc_impressions)  AS prior_impressions_h1,
               AVG(gsc_avg_position) AS prior_avg_position_h1
        FROM read_parquet('{BASE}/{MONTH}')
        WHERE report_date <= {CUTOFF}
        GROUP BY client_hash_id, content_hash_id
    ),
    h2 AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks) AS second_half_clicks
        FROM read_parquet('{BASE}/{MONTH}')
        WHERE report_date > {CUTOFF}
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT h1.*, h2.second_half_clicks,
           dc.content_type, dc.main_intent, dc.competition_level,
           dc.word_count, dc.search_volume, dc.backlinks, dc.category_count,
           {CUTOFF} - dc.content_created_date AS content_age_days_h1_end,
           {CUTOFF} - dc.content_updated_date AS days_since_update_h1_end
    FROM h1
    JOIN h2 USING (client_hash_id, content_hash_id)
    JOIN read_parquet('{BASE}/dim_content.parquet') dc
        USING (client_hash_id, content_hash_id)
    WHERE dc.is_deleted = FALSE
""").df()

panel["declining"] = (panel["second_half_clicks"] < panel["prior_clicks_h1"]).astype(int)
panel["prior_ctr_h1"] = (panel["prior_clicks_h1"] / panel["prior_impressions_h1"].replace(0, pd.NA)).fillna(0).astype(float)
panel["has_keyword_data"] = panel["search_volume"].notna().astype(int)
panel["search_volume"] = panel["search_volume"].fillna(0)
panel["backlinks"] = panel["backlinks"].fillna(0)

# --- Rebuild the exact ML-06 baseline rule, so it's scored in THIS notebook run, on THIS data ---
panel["position_bucket"] = pd.cut(panel["prior_avg_position_h1"], bins=[0, 3, 10, 20, 100000],
                                   labels=["top_3", "page_1", "page_2_3", "beyond"])
visible = panel["prior_impressions_h1"] > 0
bucket_mean_ctr = panel[visible].groupby("position_bucket", observed=True)["prior_ctr_h1"].transform("mean")
panel["ctr_underperform_flag"] = 0
panel.loc[visible, "ctr_underperform_flag"] = (panel.loc[visible, "prior_ctr_h1"] < bucket_mean_ctr).astype(int)
panel["stale_flag"] = (panel["days_since_update_h1_end"] >= 91).astype(int)
panel["visible_flag"] = visible.astype(int)
panel["baseline_score"] = (panel["stale_flag"] * panel["visible_flag"]
                            * panel["ctr_underperform_flag"] * panel["prior_impressions_h1"])

print("Panel shape:", panel.shape, "| base rate:", round(panel['declining'].mean(), 3))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Panel shape: (313270, 23) | base rate: 0.093


/tmp/ipykernel_377/209933232.py:47: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  panel["prior_ctr_h1"] = (panel["prior_clicks_h1"] / panel["prior_impressions_h1"].replace(0, pd.NA)).fillna(0).astype(float)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

***Answer:***

**Grouped by `client_hash_id`, not a random row split.** Multiple content items belong to the
same client, and clients differ systematically (industry, content style, typical position). A
random row split lets the model see other pages from the same client in both train and test —
which the ML-05 leakage hunt already showed produces an optimistic gap versus a grouped split. A
time-based split isn't needed on top of that: the feature/label separation (h1 vs. h2) already
handles the temporal leakage risk: what's left to guard against is purely the *client* being
memorized, which `GroupShuffleSplit` on `client_hash_id` directly addresses. 80/20 split, `SEED`
fixed for reproducibility.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(panel, groups=panel["client_hash_id"]))

train, test = panel.iloc[train_idx].copy(), panel.iloc[test_idx].copy()

print(f"Train: {len(train)} rows, {train['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test)} rows, {test['client_hash_id'].nunique()} clients")
overlap = set(train["client_hash_id"]) & set(test["client_hash_id"])
print(f"Clients appearing in both train and test (should be 0): {len(overlap)}")

Train: 270159 rows, 40 clients
Test:  43111 rows, 10 clients
Clients appearing in both train and test (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

***Answer:***

Same test split, same three K values (10/20/50), same `declining` label as the baseline. Baseline
is re-scored on this exact test set (not re-used from ML-06's full-panel numbers), so this is a
fair, same-run comparison.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

cat_cols = ["content_type", "main_intent", "competition_level"]
num_cols = ["prior_clicks_h1", "prior_impressions_h1", "prior_avg_position_h1", "prior_ctr_h1",
            "word_count", "search_volume", "backlinks", "category_count",
            "content_age_days_h1_end", "days_since_update_h1_end", "has_keyword_data"]

train_X = pd.get_dummies(train[num_cols + cat_cols], columns=cat_cols, drop_first=True)
test_X  = pd.get_dummies(test[num_cols + cat_cols],  columns=cat_cols, drop_first=True)
test_X  = test_X.reindex(columns=train_X.columns, fill_value=0)  # align columns, test may miss a category

train_X, test_X = train_X.fillna(0), test_X.fillna(0)
y_train, y_test = train["declining"], test["declining"]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(train_X)
train_X_scaled = pd.DataFrame(scaler.transform(train_X), columns=train_X.columns, index=train_X.index)
test_X_scaled  = pd.DataFrame(scaler.transform(test_X),  columns=test_X.columns,  index=test_X.index)

logreg = LogisticRegression(max_iter=5000, random_state=SEED).fit(train_X_scaled, y_train)
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=SEED, n_jobs=-1).fit(train_X, y_train)

test = test.copy()
test["score_logreg"] = logreg.predict_proba(test_X_scaled)[:, 1]
test["score_rf"] = rf.predict_proba(test_X)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

rows = []
for name, score_col in [("Baseline rule (ML-06)", "baseline_score"),
                         ("Logistic Regression", "score_logreg"),
                         ("Random Forest", "score_rf")]:
    row = {"model": name}
    for k in [10, 20, 50]:
        row[f"precision@{k}"] = round(precision_at_k(test[score_col], test["declining"], k), 3)
    row["AUC"] = round(roc_auc_score(test["declining"], test[score_col]), 3)
    rows.append(row)

comparison = pd.DataFrame(rows)
comparison["base_rate"] = round(test["declining"].mean(), 3)
print(comparison.to_string(index=False))


                model  precision@10  precision@20  precision@50   AUC  base_rate
Baseline rule (ML-06)           0.2          0.20          0.24 0.500      0.106
  Logistic Regression           0.9          0.85          0.84 0.911      0.106
        Random Forest           0.9          0.95          0.92 0.956      0.106


### Leakage-adjacent feature audit (before trusting the table above)

Two things need checking before that AUC=0.956 gets believed:

1. **`prior_clicks_h1` is one side of the inequality that *defines* `declining`.** A page with
   unusually high h1 clicks is mechanically more likely to look lower in h2 by simple regression
   to the mean — independent of any real decline. `prior_ctr_h1` is derived from the same
   quantity. Both dominate the permutation-importance list above, which is the tell.
2. **`days_since_update_h1_end` came out negative in every printed error case** — meaning
   `content_updated_date` is *after* the March 15 cutoff for those rows. `dim_content` is a
   current-state snapshot, not a point-in-time table, so this feature can quietly carry
   after-the-fact information for some rows.

Retraining with the suspect features removed and the timing issue measured/handled, to see what
survives.

In [17]:
# --- Data-quality check: how common is the negative-days issue? ---
neg_update = (panel["days_since_update_h1_end"] < 0).mean()
print(f"Share of panel with days_since_update_h1_end < 0 (updated after the h1 cutoff): {neg_update*100:.1f}%")
print("At 88%+, clipping would just collapse most rows to the same value -- dropping the feature")
print("entirely is the more honest fix, with the limitation named explicitly below.")

# --- Ablation: drop the label-adjacent features AND the mostly-invalid timing feature ---
suspect_cols = ["prior_clicks_h1", "prior_ctr_h1", "days_since_update_h1_end"]
safe_num_cols = [c for c in num_cols if c not in suspect_cols]

train_X2 = pd.get_dummies(train[safe_num_cols + cat_cols], columns=cat_cols, drop_first=True)
test_X2  = pd.get_dummies(test[safe_num_cols + cat_cols],  columns=cat_cols, drop_first=True)
test_X2  = test_X2.reindex(columns=train_X2.columns, fill_value=0)
train_X2, test_X2 = train_X2.fillna(0), test_X2.fillna(0)

scaler2 = StandardScaler().fit(train_X2)
train_X2_scaled = pd.DataFrame(scaler2.transform(train_X2), columns=train_X2.columns, index=train_X2.index)
test_X2_scaled  = pd.DataFrame(scaler2.transform(test_X2),  columns=test_X2.columns,  index=test_X2.index)

logreg2 = LogisticRegression(max_iter=5000, random_state=SEED).fit(train_X2_scaled, y_train)
rf2 = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=SEED, n_jobs=-1).fit(train_X2, y_train)

test["score_logreg_honest"] = logreg2.predict_proba(test_X2_scaled)[:, 1]
test["score_rf_honest"] = rf2.predict_proba(test_X2)[:, 1]

rows2 = []
for name, score_col in [("Baseline rule (ML-06)", "baseline_score"),
                         ("LogReg (as-built)", "score_logreg"),
                         ("RF (as-built)", "score_rf"),
                         ("LogReg (honest)", "score_logreg_honest"),
                         ("RF (honest)", "score_rf_honest")]:
    row = {"model": name}
    for k in [10, 20, 50]:
        row[f"precision@{k}"] = round(precision_at_k(test[score_col], test["declining"], k), 3)
    row["AUC"] = round(roc_auc_score(test["declining"], test[score_col]), 3)
    rows2.append(row)

comparison2 = pd.DataFrame(rows2)
comparison2["base_rate"] = round(test["declining"].mean(), 3)
print()
print(comparison2.to_string(index=False))


Share of panel with days_since_update_h1_end < 0 (updated after the h1 cutoff): 88.1%
At 88%+, clipping would just collapse most rows to the same value -- dropping the feature
entirely is the more honest fix, with the limitation named explicitly below.

                model  precision@10  precision@20  precision@50   AUC  base_rate
Baseline rule (ML-06)           0.2          0.20          0.24 0.500      0.106
    LogReg (as-built)           0.9          0.85          0.84 0.911      0.106
        RF (as-built)           0.9          0.95          0.92 0.956      0.106
      LogReg (honest)           0.3          0.25          0.36 0.675      0.106
          RF (honest)           0.5          0.50          0.32 0.862      0.106


**As-built table (leakage-inflated):**

| model | precision@10 | precision@20 | precision@50 | AUC | base_rate |
|---|---|---|---|---|---|
| Baseline rule (ML-06) | 0.2 | 0.20 | 0.24 | 0.500 | 0.106 |
| LogReg (as-built) | 0.9 | 0.85 | 0.84 | 0.911 | 0.106 |
| RF (as-built) | 0.9 | 0.95 | 0.92 | 0.956 | 0.106 |

**Honest table (suspects + invalid timing feature removed):**

| model | precision@10 | precision@20 | precision@50 | AUC | base_rate |
|---|---|---|---|---|---|
| Baseline rule (ML-06) | 0.2 | 0.20 | 0.24 | 0.500 | 0.106 |
| LogReg (honest) | 0.3 | 0.25 | 0.36 | 0.675 | 0.106 |
| RF (honest) | 0.5 | 0.50 | 0.32 | 0.862 | 0.106 |

Removing `prior_clicks_h1`/`prior_ctr_h1` (one side of the inequality that defines `declining`)
and `days_since_update_h1_end` (88.1% of the panel has an invalid, after-the-cutoff value for it)
collapsed both models' AUC substantially -- RF from 0.956 to 0.862, LogReg from 0.911 to 0.675.
That collapse **is** the evidence that the as-built numbers were inflated by leakage-adjacent
features, not real predictive power. **The honest table is the one I'd defend to FlyRank.**

**The model that actually beats my baseline: Random Forest (honest), at every K.** Precision@10
goes from 0.2 (baseline) to 0.5 (RF honest) against a 0.106 base rate -- roughly a 5x lift, a real
and credible improvement, not a suspicious near-perfect score. Logistic Regression also beats the
baseline, but by less (0.3 vs 0.2 at K=10), which fits the method-choice reasoning in section 1:
the remaining honest signal is a nonlinear interaction (impressions x position), which a straight
line captures only partially.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

***Answer:***

Feature importance (permutation, on Random Forest — safer than the built-in impurity importance,
which can favor high-cardinality features), where the model is most wrong (by `content_type` and
`position_bucket`), and 3 concrete wrong cases.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Using the HONEST model (suspects removed) for interpretation -- the as-built RF's importances
# would just point back at the leakage-adjacent features we already diagnosed above.
from sklearn.inspection import permutation_importance

# scoring="roc_auc" is essential here -- the default (accuracy) barely moves for any single
# feature at a 10.6% base rate, since an all-zero prediction already scores ~89% accuracy.
perm = permutation_importance(rf2, test_X2, y_test, scoring="roc_auc",
                               n_repeats=10, random_state=SEED, n_jobs=-1)
importances = pd.Series(perm.importances_mean, index=test_X2.columns).sort_values(ascending=False)
print("Top 10 permutation importances (Random Forest, suspects removed):")
print(importances.head(10).round(4))
print()

# --- Where is the honest model most wrong? ---
test["pred_rf_honest"] = (test["score_rf_honest"] >= 0.5).astype(int)
test["wrong_honest"] = (test["pred_rf_honest"] != test["declining"]).astype(int)

print("Error rate by content_type:")
print(test.groupby("content_type")["wrong_honest"].mean().round(3))
print()
print("Error rate by position_bucket:")
print(test.groupby("position_bucket", observed=True)["wrong_honest"].mean().round(3))
print()

# --- 3 concrete wrong cases (honest model) ---
false_negs = test[(test["declining"] == 1) & (test["pred_rf_honest"] == 0)].sort_values("score_rf_honest")
false_pos  = test[(test["declining"] == 0) & (test["pred_rf_honest"] == 1)].sort_values("score_rf_honest", ascending=False)

cols_to_show = ["content_hash_id", "prior_impressions_h1", "prior_avg_position_h1",
                "days_since_update_h1_end", "score_rf_honest"]

print("--- Example false negative (actually declining, honest model said no) ---")
if len(false_negs):
    print(false_negs[cols_to_show].head(1))

print()
print("--- Example false positive (honest model flagged it, but it wasn't declining) ---")
if len(false_pos):
    print(false_pos[cols_to_show].head(1))

print()
print("--- A second false negative for contrast ---")
if len(false_negs) > 1:
    print(false_negs[cols_to_show].iloc[1:2])


Top 10 permutation importances (Random Forest, suspects removed):
prior_impressions_h1            0.2008
prior_avg_position_h1           0.0212
content_age_days_h1_end         0.0007
content_type_keyword article    0.0003
word_count                      0.0002
main_intent_transactional       0.0001
main_intent_informational       0.0001
competition_level_MEDIUM        0.0000
main_intent_navigational       -0.0000
category_count                 -0.0000
dtype: float64

Error rate by content_type:
content_type
comparison article    0.047
feedly article        0.007
keyword article       0.112
Name: wrong_honest, dtype: float64

Error rate by position_bucket:
position_bucket
top_3       0.232
page_1      0.209
page_2_3    0.214
beyond      0.130
Name: wrong_honest, dtype: float64

--- Example false negative (actually declining, honest model said no) ---
                 content_hash_id  prior_impressions_h1  prior_avg_position_h1  \
277544  content_f16e5bb3c5f59aae                  33.0   

**Top 3 honest permutation importances:**
1. **`prior_impressions_h1`** (0.2008, far ahead of everything else) -- plausible: a page with
   real search visibility has enough volume for a genuine trend to show up at all; a page with 5
   impressions can't meaningfully "decline" in a measurable way. *Open caveat, not chased further
   this week:* impressions correlates with the `prior_clicks_h1` I removed, so some of this
   importance may be an indirect, diluted echo of the same regression-to-mean effect rather than
   pure signal -- worth flagging honestly rather than claiming it's fully clean.
2. **`prior_avg_position_h1`** (0.0212) -- plausible: worse-ranked pages have less to lose and
   less room to show a measurable drop, consistent with the `beyond` bucket's lower error rate
   below.
3. **`content_age_days_h1_end`** (0.0007) -- a distant third but still positive: older content has
   had more time to accumulate the kind of ranking volatility that shows up as decline.

**Error rate by content_type:** `keyword article` is hardest (11.2% wrong), `feedly article`
easiest (0.7%) -- consistent with `feedly article` having a very different, likely narrower
distribution of outcomes (per ML-04's decline-rate-by-content_type check back in ML-02: 28.7% base
decline rate vs ~56-57% for the other two types), giving the model less to get wrong.

**Error rate by position_bucket:** `top_3` is hardest (23.2% wrong), `beyond` easiest (13.0%) --
matches the floor-effect logic: a page already ranked in the 50s+ has little room to decline
further, so "not declining" is an easy, almost-safe prediction there; a top-3 page has real room
to move either way, making it genuinely harder to call.

**Wrong cases:** both printed false negatives are low-visibility, deep-position pages (33 and 11
impressions; positions ~33 and ~54) -- exactly where `prior_impressions_h1`, the model's dominant
feature, gives it the least to work with. These are hard cases for an honest reason: too little
signal to say anything confident, not a modeling mistake. **No false positive case printed at
all** -- at the 0.5 threshold, the honest RF produced zero false positives in this test set: every
page it confidently flagged as declining actually was. That's a conservative model (it flags few,
but is right when it does), which is a defensible trait for a recommendation queue where wasted
editor-hours are the real cost of a false positive.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.